# 🏎️ F1 Race Outcome Predictor
## Notebook 01 — Data Collection

**Project:** Predict Formula 1 race outcomes using historical race data, telemetry, and weather conditions.  
**Author:** Johar Rizvi  

---

### What this notebook does
Collects data from three sources and merges them into one master dataset ready for EDA and ML.

| Source | What it gives us | Years |
|---|---|---|
| **Jolpica API** | Race results, qualifying times, pit stops | 2022–2025 |
| **FastF1** | Lap telemetry, tyre strategy, weather | 2022–2025 |
| **OpenF1 API** | Driver nationality (country_code) | 2023–2025 |

### Output
`../data/processed/master_dataset.csv` — 1838 rows × 76 columns, one row per driver per race.

### How to run
- **First time:** Run all cells top to bottom. FastF1 downloads ~50MB per race and caches locally. Expect 1–2 hours total due to API rate limits.
- **Subsequent runs:** Data is saved to `../data/raw/`. Skip collection cells and jump straight to the merge section.

### Data sources
- Jolpica: https://github.com/jolpica/jolpica-f1 (Ergast API replacement)
- FastF1: https://docs.fastf1.dev
- OpenF1: https://openf1.org

---
## Section 1 — Imports and Setup

In [ ]:
import os
import time
import json
import warnings
warnings.filterwarnings('ignore')

import requests
import pandas as pd
import numpy as np
import fastf1

# ── API base URLs ──────────────────────────────────────────
BASE_URL    = 'https://api.jolpi.ca/ergast/f1'   # Jolpica (Ergast replacement)
OPENF1_BASE = 'https://api.openf1.org/v1'         # OpenF1

# ── FastF1 cache ───────────────────────────────────────────
# FastF1 downloads data from F1 servers on first load,
# then caches locally so subsequent loads are instant.
CACHE_DIR = '../f1_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
fastf1.Cache.enable_cache(CACHE_DIR)

# ── Seasons to collect ─────────────────────────────────────
# 2022 onwards = modern ground-effect car regulations era.
# Older data describes a different sport and would hurt model accuracy.
YEARS = [2022, 2023, 2024, 2025]

print('✅ Setup complete')
print(f'   FastF1 version: {fastf1.__version__}')
print(f'   Collecting seasons: {YEARS}')

---
## Section 2 — Jolpica API

Jolpica is the community-maintained replacement for the deprecated Ergast API.
It returns the same JSON format, just at a different base URL.

We collect three things:
- **Race results** — finishing position, grid position, points, status, constructor
- **Qualifying times** — Q1/Q2/Q3 times and qualifying position
- **Pit stops** — lap of stop, duration, total stops per driver per race

**Rate limits:** The API is polite about usage. We sleep 0.4s between calls.

In [ ]:
# ══════════════════════════════════════════════════════════
# WHAT:  Fetch all race results for one season
# WHY:   Race results are the backbone of the dataset —
#        finishing positions are our primary prediction target
# INPUT: year (int)
# PLAN:
#   Step 1 → call Jolpica API with pagination (max 100 rows per call)
#   Step 2 → loop over all pages until we have everything
#   Step 3 → flatten nested JSON into one row per driver per race
# OUTPUT: DataFrame — one row per driver per race
# ══════════════════════════════════════════════════════════

def get_race_results(year: int) -> pd.DataFrame:
    """Fetch all race results for one season from Jolpica."""
    rows   = []
    offset = 0
    limit  = 100

    while True:
        url      = f'{BASE_URL}/{year}/results.json?limit={limit}&offset={offset}'
        response = requests.get(url)

        if response.status_code != 200:
            print(f'  ERROR {year}: {response.status_code}')
            break

        mrdata = response.json()['MRData']
        races  = mrdata['RaceTable']['Races']

        if not races:
            break

        for race in races:
            for result in race['Results']:
                driver      = result['Driver']
                constructor = result['Constructor']
                rows.append({
                    # Race context
                    'year':             year,
                    'round':            int(race['round']),
                    'race_name':        race['raceName'],
                    'circuit_id':       race['Circuit']['circuitId'],
                    'circuit_name':     race['Circuit']['circuitName'],
                    'country':          race['Circuit']['Location']['country'],
                    'date':             race['date'],
                    # Driver
                    'driver_id':        driver['driverId'],
                    'driver_code':      driver.get('code', 'N/A'),
                    'driver_name':      f"{driver['givenName']} {driver['familyName']}",
                    # Constructor
                    'constructor_id':   constructor['constructorId'],
                    'constructor_name': constructor['name'],
                    # Race performance
                    'grid_position':    int(result['grid']),
                    'finish_position':  int(result['position']),
                    'points':           float(result['points']),
                    'laps_completed':   int(result['laps']),
                    'status':           result['status'],
                    # Derived ML target variables
                    'finished':         result['status'] == 'Finished',
                    'on_podium':        int(result['position']) <= 3,
                    'points_finish':    float(result['points']) > 0,
                })

        total_available = int(mrdata['total'])
        offset += limit
        if offset >= total_available:
            break

        time.sleep(0.5)

    return pd.DataFrame(rows)


print('Collecting race results...')
all_results = []

for year in YEARS:
    print(f'  {year}...', end=' ')
    df_year = get_race_results(year)
    all_results.append(df_year)
    print(f'✅ {len(df_year)} rows ({df_year["round"].nunique()} races)')
    time.sleep(0.5)

df_results = pd.concat(all_results, ignore_index=True)
df_results['date'] = pd.to_datetime(df_results['date'])

df_results.to_csv('../data/raw/jolpica_results.csv', index=False)
print(f'\n📦 Results: {df_results.shape} → saved to ../data/raw/jolpica_results.csv')

In [ ]:
# ══════════════════════════════════════════════════════════
# WHAT:  Fetch qualifying times for all seasons
# WHY:   Qualifying position is one of the strongest
#        predictors of race outcome — we always want it
# ══════════════════════════════════════════════════════════

def get_qualifying_results(year: int) -> pd.DataFrame:
    """Fetch qualifying results for one season from Jolpica."""
    rows   = []
    offset = 0
    limit  = 100

    while True:
        url      = f'{BASE_URL}/{year}/qualifying.json?limit={limit}&offset={offset}'
        response = requests.get(url)

        if response.status_code != 200:
            break

        mrdata = response.json()['MRData']
        races  = mrdata['RaceTable']['Races']

        if not races:
            break

        for race in races:
            for result in race['QualifyingResults']:
                driver = result['Driver']
                rows.append({
                    'year':          year,
                    'round':         int(race['round']),
                    'driver_id':     driver['driverId'],
                    'qual_position': int(result['position']),
                    'q1_time':       result.get('Q1', None),
                    'q2_time':       result.get('Q2', None),
                    'q3_time':       result.get('Q3', None),
                })

        total_available = int(mrdata['total'])
        offset += limit
        if offset >= total_available:
            break

        time.sleep(0.5)

    return pd.DataFrame(rows)


print('Collecting qualifying results...')
all_quali = []

for year in YEARS:
    print(f'  {year}...', end=' ')
    df_q = get_qualifying_results(year)
    all_quali.append(df_q)
    print(f'✅ {len(df_q)} rows')
    time.sleep(0.5)

df_quali = pd.concat(all_quali, ignore_index=True)
df_quali.to_csv('../data/raw/jolpica_qualifying.csv', index=False)
print(f'\n📦 Qualifying: {df_quali.shape} → saved to ../data/raw/jolpica_qualifying.csv')

In [ ]:
# ══════════════════════════════════════════════════════════
# WHAT:  Fetch pit stop data for all seasons
# WHY:   Pit stop strategy is a major factor in race outcomes.
#        Number of stops, timing, and duration all matter.
# NOTE:  Pit stop endpoint requires one API call per race
#        (unlike results which gives a full season per call),
#        so this takes longer — ~20 minutes for 4 seasons.
# ══════════════════════════════════════════════════════════

def get_pit_stops(year: int, round_num: int) -> pd.DataFrame:
    """Fetch all pit stops for one race. Has retry logic for connection errors."""
    url = f'{BASE_URL}/{year}/{round_num}/pitstops.json?limit=100'

    for attempt in range(3):   # try up to 3 times
        try:
            response = requests.get(url, timeout=10)

            if response.status_code != 200:
                return pd.DataFrame()

            races = response.json()['MRData']['RaceTable']['Races']
            if not races:
                return pd.DataFrame()

            rows = []
            for stop in races[0].get('PitStops', []):
                rows.append({
                    'year':         year,
                    'round':        round_num,
                    'driver_id':    stop['driverId'],
                    'stop_number':  int(stop['stop']),
                    'stop_lap':     int(stop['lap']),
                    'duration_sec': stop['duration'],
                })
            return pd.DataFrame(rows)

        except Exception as e:
            wait = (attempt + 1) * 10
            print(f'\n  ⚠️  Round {round_num} attempt {attempt+1} failed. Waiting {wait}s...')
            time.sleep(wait)

    print(f'\n  ❌ Round {round_num} skipped after 3 attempts')
    return pd.DataFrame()


print('Collecting pit stops (~20 mins — one API call per race)...')
pit_dfs = []

for year in YEARS:
    n_rounds = int(df_results[df_results['year'] == year]['round'].max())
    print(f'\n{year} ({n_rounds} rounds): ', end='', flush=True)

    for r in range(1, n_rounds + 1):
        df_p = get_pit_stops(year, r)
        if not df_p.empty:
            pit_dfs.append(df_p)
        print('·', end='', flush=True)
        time.sleep(1.5)

    print(' ✅')
    time.sleep(30)   # rest between years

# Raw pit stops — one row per stop event
df_pitstops_raw = pd.concat(pit_dfs, ignore_index=True)
print(f'\nRaw pit stops: {df_pitstops_raw.shape}')

# Aggregate to one row per driver per race
df_pitstops_agg = df_pitstops_raw.groupby(
    ['year', 'round', 'driver_id']
).agg(
    total_pit_stops  = ('stop_number', 'max'),
    avg_pit_duration = ('duration_sec', 'mean'),
    first_stop_lap   = ('stop_lap',     'min'),
).reset_index()

df_pitstops_agg.to_csv('../data/raw/jolpica_pitstops_agg.csv', index=False)
print(f'📦 Pitstops aggregated: {df_pitstops_agg.shape} → saved to ../data/raw/jolpica_pitstops_agg.csv')

---
## Section 3 — FastF1

FastF1 is a Python library that downloads official F1 timing data and parses it into DataFrames.
It gives us **much richer data** than Jolpica — lap-by-lap telemetry, tyre compounds, and weather.

**How it works:**
- `fastf1.get_session(year, round, 'R')` creates a Session object for one race
- `session.load()` downloads the data (cached locally after first download)
- `session.laps` → DataFrame of every lap driven by every driver
- `session.weather_data` → weather readings sampled throughout the race

**Rate limits:** FastF1 pulls from F1's timing servers which allow ~500 calls/hour.
We sleep 5s between rounds and 120s between seasons to stay well within the limit.
Once cached, re-runs are instant — no API calls needed.

In [ ]:
# ══════════════════════════════════════════════════════════
# WHAT:  Helper functions for FastF1 data extraction
# ══════════════════════════════════════════════════════════

def extract_lap_features(session) -> pd.DataFrame:
    """
    Extract and clean lap-level features from one FastF1 session.
    Returns a DataFrame with one row per lap per driver.
    """
    laps = session.laps.copy()   # copy so we don't modify FastF1's original

    # Convert all Timedelta columns → seconds (float)
    # ML models need plain numbers, not Timedelta objects
    timedelta_cols = [
        col for col in laps.columns
        if pd.api.types.is_timedelta64_dtype(laps[col])
    ]
    for col in timedelta_cols:
        laps[f'{col}_s'] = laps[col].dt.total_seconds()

    # Flag valid laps (not pit in/out laps, not safety car laps)
    # IsAccurate is FastF1's own flag for reliable timing
    laps['is_valid_lap'] = (
        laps['IsAccurate'] &
        laps['LapTime_s'].notna() &
        (laps['LapTime_s'] > 0)
    )

    # Stamp each lap with its race context
    # (needed when stacking many races together)
    laps['year']  = session.event.year
    laps['round'] = session.event['RoundNumber']
    laps['event'] = session.event['EventName']

    # Put context columns first, keep everything else after
    priority_cols = [
        'year', 'round', 'event', 'Driver', 'Team',
        'LapNumber', 'LapTime_s',
        'Sector1Time_s', 'Sector2Time_s', 'Sector3Time_s',
        'Compound', 'TyreLife', 'Stint',
        'is_valid_lap', 'IsPersonalBest', 'TrackStatus',
    ]
    front = [c for c in priority_cols if c in laps.columns]
    rest  = [c for c in laps.columns if c not in front]
    return laps[front + rest]


def aggregate_weather(session) -> dict:
    """
    Summarise a race's weather into one dictionary (one row later).
    Uses a config-driven loop so adding new stats requires only
    editing the numeric_cols dict.
    """
    w = session.weather_data

    result = {
        'year':  session.event.year,
        'round': session.event['RoundNumber'],
    }

    # For each column, compute these summary statistics
    numeric_cols = {
        'AirTemp':       ['mean', 'min', 'max'],
        'TrackTemp':     ['mean', 'min', 'max'],
        'Humidity':      ['mean', 'max'],
        'WindSpeed':     ['mean', 'max'],
        'WindDirection': ['mean'],
        'Pressure':      ['mean', 'min', 'max'],
    }

    for col, stats in numeric_cols.items():
        for stat in stats:
            result[f'{col.lower()}_{stat}'] = getattr(w[col], stat)()

    # Range = how much did conditions change during the race
    for col in ['TrackTemp', 'AirTemp', 'WindSpeed']:
        result[f'{col.lower()}_range'] = w[col].max() - w[col].min()

    # Rainfall — keep full picture, not just a boolean
    result['had_rain']      = bool(w['Rainfall'].any())
    result['rain_laps']     = int(w['Rainfall'].sum())
    result['rain_fraction'] = float(w['Rainfall'].mean())

    return result


print('✅ FastF1 helper functions defined')

In [ ]:
# ══════════════════════════════════════════════════════════
# WHAT:  Collect FastF1 lap and weather data for all seasons
# WHY:   FastF1 gives us pace data (lap times, sector splits,
#        tyre info) that Jolpica doesn't have
# NOTE:  Run one year at a time to stay within rate limits.
#        Change YEAR below and re-run for each season.
#        All data is cached — re-running is instant.
# ══════════════════════════════════════════════════════════

def collect_fastf1_season(year: int) -> tuple:
    """
    Collect all lap and weather data for one full season.
    Tries up to round 25 and skips rounds that don't exist.
    Returns (df_laps, df_weather).
    """
    laps_list    = []
    weather_list = []

    print(f'Collecting FastF1 {year}...')

    for rnd in range(1, 26):   # F1 never exceeds 25 rounds
        try:
            sess = fastf1.get_session(year, rnd, 'R')
            sess.load(telemetry=False, weather=True)
            laps_list.append(extract_lap_features(sess))
            weather_list.append(aggregate_weather(sess))
            print(f'  Round {rnd:2d} - {sess.event["EventName"]:30s} ✅')
        except Exception as e:
            print(f'  Round {rnd:2d} - skipping ({e})')

        time.sleep(5)   # 5s between rounds

    df_laps    = pd.concat(laps_list,    ignore_index=True) if laps_list    else pd.DataFrame()
    df_weather = pd.DataFrame(weather_list)                  if weather_list else pd.DataFrame()

    print(f'  → laps: {df_laps.shape}, weather: {df_weather.shape}')
    return df_laps, df_weather


# Collect all years and append to CSVs
# If the CSV already exists, we append to it rather than re-collecting
for year in YEARS:
    df_laps_year, df_weather_year = collect_fastf1_season(year)

    # Append to existing file if it exists, otherwise create new
    laps_path    = '../data/raw/fastf1_laps.csv'
    weather_path = '../data/raw/fastf1_weather.csv'

    if not df_laps_year.empty:
        if os.path.exists(laps_path):
            existing = pd.read_csv(laps_path)
            # Only add rounds we don't already have
            existing_rounds = set(zip(existing['year'], existing['round']))
            new_rounds = df_laps_year[~df_laps_year.apply(
                lambda r: (r['year'], r['round']) in existing_rounds, axis=1
            )]
            if not new_rounds.empty:
                pd.concat([existing, new_rounds], ignore_index=True).to_csv(laps_path, index=False)
                print(f'  ✅ Added {new_rounds["round"].nunique()} new rounds to fastf1_laps.csv')
            else:
                print(f'  ℹ️  {year} already fully in fastf1_laps.csv')
        else:
            df_laps_year.to_csv(laps_path, index=False)

    if not df_weather_year.empty:
        if os.path.exists(weather_path):
            existing_w = pd.read_csv(weather_path)
            existing_rounds_w = set(zip(existing_w['year'], existing_w['round']))
            new_weather = df_weather_year[~df_weather_year.apply(
                lambda r: (r['year'], r['round']) in existing_rounds_w, axis=1
            )]
            if not new_weather.empty:
                pd.concat([existing_w, new_weather], ignore_index=True).to_csv(weather_path, index=False)
        else:
            df_weather_year.to_csv(weather_path, index=False)

    print(f'  Waiting 2 minutes before next season...\n')
    time.sleep(120)

# Verify
df_laps    = pd.read_csv('../data/raw/fastf1_laps.csv', low_memory=False)
df_weather = pd.read_csv('../data/raw/fastf1_weather.csv')
print(f'\n📦 FastF1 laps:    {df_laps.shape}')
print(f'📦 FastF1 weather: {df_weather.shape}')
print(f'Races per year (laps):')
print(df_laps.groupby('year')['round'].nunique().to_string())

---
## Section 4 — OpenF1 API

OpenF1 is the modern, actively maintained F1 data API (2023 onwards).
Historical data is free with no authentication required.

We use it for one thing Jolpica doesn't give us: **driver nationality** (`country_code`).
Home race advantage is a real phenomenon in F1 — drivers often perform better
at their home Grand Prix due to crowd support and familiarity.

The challenge: OpenF1 doesn't use round numbers — it uses `session_key` and `meeting_key`.
We bridge this to Jolpica's round numbers via country name matching.

In [ ]:
# ══════════════════════════════════════════════════════════
# WHAT:  Collect driver nationality from OpenF1
# WHY:   country_code (driver nationality) is not in Jolpica
#        and may be a useful predictor for home race advantage
# ══════════════════════════════════════════════════════════

def get_openf1_sessions(year: int) -> pd.DataFrame:
    """Get all Race sessions for a given year from OpenF1."""
    url      = f'{OPENF1_BASE}/sessions?year={year}&session_name=Race'
    response = requests.get(url)

    if response.status_code != 200:
        print(f'  ERROR {year}: API error {response.status_code}')
        return pd.DataFrame()

    sessions = response.json()
    if not sessions:
        return pd.DataFrame()

    print(f'  {year}: {len(sessions)} sessions found')
    return pd.DataFrame(sessions)


def get_openf1_drivers(session_key: int) -> pd.DataFrame:
    """Get driver info for one session — keep all columns."""
    url      = f'{OPENF1_BASE}/drivers?session_key={session_key}'
    response = requests.get(url)

    if response.status_code != 200:
        return pd.DataFrame()

    drivers = response.json()
    return pd.DataFrame(drivers) if drivers else pd.DataFrame()


def collect_openf1_nationality(years: list) -> pd.DataFrame:
    """
    Collect driver nationality (country_code) from OpenF1.
    Returns DataFrame with year, race_name, driver code, country_code.
    """
    all_rows = []

    for year in years:
        print(f'\nCollecting OpenF1 {year}...')
        df_sessions = get_openf1_sessions(year)

        if df_sessions.empty:
            continue

        for _, session in df_sessions.iterrows():
            sk         = session['session_key']
            race_name  = session['country_name']

            df_drv = get_openf1_drivers(sk)
            if df_drv.empty:
                continue

            # Keep only what we need
            if 'name_acronym' in df_drv.columns and 'country_code' in df_drv.columns:
                subset = df_drv[['name_acronym', 'country_code']].copy()
                subset['year']      = year
                subset['race_name'] = race_name
                all_rows.append(subset)

            time.sleep(0.5)

        time.sleep(5)

    if not all_rows:
        return pd.DataFrame()

    df = pd.concat(all_rows, ignore_index=True)
    df = df.rename(columns={'name_acronym': 'driver_code'})
    return df


# OpenF1 only has reliable data from 2023 onwards
OPENF1_YEARS = [2023, 2024, 2025]
df_openf1 = collect_openf1_nationality(OPENF1_YEARS)

if not df_openf1.empty:
    df_openf1.to_csv('../data/raw/openf1_results.csv', index=False)
    print(f'\n📦 OpenF1: {df_openf1.shape} → saved to ../data/raw/openf1_results.csv')
else:
    print('\n⚠️  OpenF1 collection failed — API may be temporarily unavailable')
    print('   The master dataset will be built without OpenF1 (country_code will be NaN)')

---
## Section 5 — Build Master Dataset

We now join all sources into one flat table for ML.

**The join strategy:**
```
df_results (Jolpica)  ← SPINE: one row per driver per race
    + df_quali        ← join on year + round + driver_id
    + df_pitstops_agg ← join on year + round + driver_id
    + df_fastf1_agg   ← join on year + round + driver_code
    + df_weather      ← join on year + round (same for all drivers)
    + df_openf1       ← join on year + round + driver_code
```

**`how='left'`** means we keep all rows from df_results even if there's no matching
row in the other table — missing values become NaN and are handled in feature engineering.

**Coverage targets:**
- Pit stops: ~95% (a few races have missing data in Jolpica)
- FastF1: ~95% (all 4 seasons collected)
- Weather: ~100% (one row per race, should match perfectly)
- OpenF1: ~50% (only covers 2023 onwards)

In [ ]:
import pandas as pd
import os

# ── Load from disk ─────────────────────────────────────────
print('Loading from disk...')
df_results      = pd.read_csv('../data/raw/jolpica_results.csv')
df_quali        = pd.read_csv('../data/raw/jolpica_qualifying.csv')
df_pitstops_agg = pd.read_csv('../data/raw/jolpica_pitstops_agg.csv')
df_laps         = pd.read_csv('../data/raw/fastf1_laps.csv', low_memory=False)
df_weather      = pd.read_csv('../data/raw/fastf1_weather.csv')
df_openf1       = pd.read_csv('../data/raw/openf1_results.csv')

print(f'  results:   {df_results.shape}')
print(f'  quali:     {df_quali.shape}')
print(f'  pitstops:  {df_pitstops_agg.shape}')
print(f'  laps:      {df_laps.shape}')
print(f'  weather:   {df_weather.shape}')
print(f'  openf1:    {df_openf1.shape}')

# ── Aggregate FastF1 laps → one row per driver per race ────
print('\nAggregating FastF1 laps...')
valid_laps = df_laps[df_laps['is_valid_lap'] == True].copy()
valid_laps['LapTime_s'] = pd.to_numeric(valid_laps['LapTime_s'], errors='coerce')
valid_laps = valid_laps[valid_laps['LapTime_s'].between(60, 200)]

df_fastf1_agg = valid_laps.groupby(['year', 'round', 'Driver']).agg(
    avg_lap_time     = ('LapTime_s', 'mean'),
    best_lap_time    = ('LapTime_s', 'min'),
    std_lap_time     = ('LapTime_s', 'std'),
    total_valid_laps = ('LapTime_s', 'count'),
    laps_on_soft     = ('Compound', lambda x: (x == 'SOFT').sum()),
    laps_on_medium   = ('Compound', lambda x: (x == 'MEDIUM').sum()),
    laps_on_hard     = ('Compound', lambda x: (x == 'HARD').sum()),
    avg_tyre_life    = ('TyreLife', 'mean'),
    total_stints     = ('Stint', 'nunique'),
).reset_index().rename(columns={'Driver': 'driver_code'})
print(f'  FastF1 aggregated: {df_fastf1_agg.shape}')

# ── Prepare OpenF1 ─────────────────────────────────────────
# Fix country name mismatches between OpenF1 and Jolpica
country_name_fixes = {
    'United Kingdom':       'UK',
    'United Arab Emirates': 'UAE',
    'United States':        'USA',
}
df_openf1_clean = df_openf1.copy()
df_openf1_clean['race_name'] = df_openf1_clean['race_name'].replace(country_name_fixes)

# Add round number by matching year + country name to Jolpica
round_lookup = (
    df_results[['year', 'round', 'country']]
    .drop_duplicates()
    .rename(columns={'country': 'race_name'})
)
df_openf1_clean = df_openf1_clean.merge(
    round_lookup, on=['year', 'race_name'], how='left'
)

# Drop rows where round couldn't be matched (2026 races not in Jolpica)
df_openf1_clean = df_openf1_clean.dropna(subset=['round'])
df_openf1_clean['round'] = df_openf1_clean['round'].astype(int)

# Extract only what OpenF1 uniquely adds — driver nationality
# name_acronym is the 3-letter driver code in OpenF1
openf1_useful = (
    df_openf1_clean[['year', 'round', 'name_acronym', 'country_code']]
    .drop_duplicates()
    .rename(columns={'name_acronym': 'driver_code'})
)
print(f'  OpenF1 useful: {openf1_useful.shape}')

# ── Merge helper ───────────────────────────────────────────
def merge_keeping_all(df_master, df_new, join_keys):
    """
    Merge df_new into df_master.
    Drops columns from df_new that already exist in df_master
    (except join keys) to prevent _x _y duplicate columns.
    """
    duplicate_cols = [
        c for c in df_new.columns
        if c in df_master.columns and c not in join_keys
    ]
    if duplicate_cols:
        print(f'    Dropping duplicates: {duplicate_cols}')
    df_new_clean = df_new.drop(columns=duplicate_cols, errors='ignore')
    return df_master.merge(df_new_clean, on=join_keys, how='left')

# ── Build master ───────────────────────────────────────────
print('\nBuilding master dataset...')
df_master = df_results.copy()
print(f'  Start:        {df_master.shape}')

df_master = merge_keeping_all(df_master, df_quali,        ['year', 'round', 'driver_id'])
print(f'  + qualifying: {df_master.shape}')

df_master = merge_keeping_all(df_master, df_pitstops_agg, ['year', 'round', 'driver_id'])
print(f'  + pitstops:   {df_master.shape}')

df_master = merge_keeping_all(df_master, df_fastf1_agg,   ['year', 'round', 'driver_code'])
print(f'  + fastf1:     {df_master.shape}')

df_master = merge_keeping_all(df_master, df_weather,      ['year', 'round'])
print(f'  + weather:    {df_master.shape}')

df_master = merge_keeping_all(df_master, openf1_useful,   ['year', 'round', 'driver_code'])
print(f'  + openf1:     {df_master.shape}')

# ── Sanity check ───────────────────────────────────────────
print('\n=== SANITY CHECK ===')
print(f'Rows:    {df_master.shape[0]}   ← must be ~1838')
print(f'Columns: {df_master.shape[1]}')
print(f'Years:   {sorted(df_master["year"].unique())}')
print(f'Races:   {df_master.groupby("year")["round"].nunique().to_dict()}')
print(f'Drivers: {df_master["driver_id"].nunique()}')

pit_pct = df_master['total_pit_stops'].notna().mean() * 100
f1_pct  = df_master['avg_lap_time'].notna().mean() * 100
of1_pct = df_master['country_code'].notna().mean() * 100

# Weather — use whichever column exists
weather_col = 'airtemp_mean' if 'airtemp_mean' in df_master.columns else 'mean_airtemp'
wth_pct = df_master[weather_col].notna().mean() * 100 if weather_col in df_master.columns else 0

print(f'\nCoverage:')
print(f'  Pit stops:  {pit_pct:.1f}%')
print(f'  FastF1:     {f1_pct:.1f}%')
print(f'  Weather:    {wth_pct:.1f}%')
print(f'  OpenF1:     {of1_pct:.1f}%')

print('\nSample:')
print(df_master[['year', 'round', 'driver_name', 'finish_position',
                  'total_pit_stops', 'avg_lap_time', 'country_code']].head(5).to_string())

# ── Save ───────────────────────────────────────────────────
os.makedirs('../data/processed', exist_ok=True)
df_master.to_csv('../data/processed/master_dataset.csv', index=False)
print(f'\n✅ master_dataset.csv saved — {df_master.shape}')
print('🏁 DATA COLLECTION COMPLETE — ready for EDA')

---
## Section 6 — Verify and Save

In [ ]:
# ══════════════════════════════════════════════════════════
# WHAT:  Sanity check the master dataset before saving
# WHY:   Catch data quality issues now rather than
#        discovering them halfway through modelling
# ══════════════════════════════════════════════════════════

print('=== MASTER DATASET SUMMARY ===')
print(f'Rows:           {df_master.shape[0]}')
print(f'Columns:        {df_master.shape[1]}')
print(f'Years:          {sorted(df_master["year"].unique())}')
print(f'Races per year: {df_master.groupby("year")["round"].nunique().to_dict()}')
print(f'Unique drivers: {df_master["driver_id"].nunique()}')

# Coverage — what % of rows have data from each source
print('\nData coverage per source:')
coverage_cols = {
    'Pit stops (Jolpica)':  'total_pit_stops',
    'Lap times (FastF1)':   'avg_lap_time',
    'Tyre data (FastF1)':   'avg_tyre_life',
    'Driver nationality':   'country_code',
}
for label, col in coverage_cols.items():
    if col in df_master.columns:
        pct = df_master[col].notna().mean() * 100
        bar = '█' * int(pct // 5)
        print(f'  {label:25s} {bar:20s} {pct:.1f}%')

# Missing values overview
print('\nColumns with missing values:')
missing = df_master.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing.to_string())

# Sample rows — verify data looks realistic
print('\nSample rows (Bahrain 2022 top 5):')
sample_cols = [
    'year', 'round', 'driver_name', 'constructor_name',
    'grid_position', 'finish_position', 'points',
    'total_pit_stops', 'avg_lap_time', 'had_rain'
]
print(df_master[sample_cols].head(5).to_string(index=False))

# Save
df_master.to_csv('../data/processed/master_dataset.csv', index=False)
print(f'\n✅ Saved: ../data/processed/master_dataset.csv — {df_master.shape}')
print('\n🏁 DATA COLLECTION COMPLETE')
print('   Next: open 02_Exploratory_Data_Analysis.ipynb')

In [ ]:
# ══════════════════════════════════════════════════════════
# FIX: Standardise weather column names
# Old collection used: mean_airtemp, max_airtemp etc.
# New collection used: airtemp_mean, airtemp_max etc.
# We unify both into one consistent set and drop the old names
# ══════════════════════════════════════════════════════════

df_master = pd.read_csv('../data/processed/master_dataset.csv')

# Map: unified name → (new column, old column fallback)
weather_fixes = {
    'air_temp_mean':    ('airtemp_mean',    'mean_airtemp'),
    'air_temp_min':     ('airtemp_min',     'min_airtemp'),
    'air_temp_max':     ('airtemp_max',     'max_airtemp'),
    'track_temp_mean':  ('tracktemp_mean',  'mean_tracktemp'),
    'track_temp_min':   ('tracktemp_min',   'min_tracktemp'),
    'track_temp_max':   ('tracktemp_max',   'max_tracktemp'),
    'humidity_mean':    ('humidity_mean',   'mean_humidity'),
    'humidity_max':     ('humidity_max',    'max_humidity'),
    'wind_speed_mean':  ('windspeed_mean',  'mean_windspeed'),
    'wind_speed_max':   ('windspeed_max',   'max_windspeed'),
    'pressure_mean':    ('pressure_mean',   'mean_pressure'),
    'pressure_min':     ('pressure_min',    'min_pressure'),
    'pressure_max':     ('pressure_max',    'max_pressure'),
}

for unified_name, (new_col, old_col) in weather_fixes.items():
    new_exists = new_col in df_master.columns
    old_exists = old_col in df_master.columns

    if new_exists and old_exists:
        df_master[unified_name] = df_master[new_col].fillna(df_master[old_col])
    elif new_exists:
        df_master[unified_name] = df_master[new_col]
    elif old_exists:
        df_master[unified_name] = df_master[old_col]

# Drop all the old inconsistent columns
old_cols_to_drop = [c for c in df_master.columns if c.startswith('mean_') 
                    or c.startswith('min_') 
                    or c.startswith('max_')
                    or c.startswith('airtemp_')
                    or c.startswith('tracktemp_')
                    or c.startswith('windspeed_')
                    or c.startswith('humidity_')
                    or c.startswith('windspeed_')
                    or c.startswith('pressure_')]

df_master = df_master.drop(columns=old_cols_to_drop, errors='ignore')

# Verify
weather_pct = df_master['air_temp_mean'].notna().mean() * 100
print(f'Weather coverage after fix: {weather_pct:.1f}%  ← should be ~100%')
print(f'Master shape: {df_master.shape}')

# Save fixed version
df_master.to_csv('../data/processed/master_dataset.csv', index=False)
print('✅ master_dataset.csv updated with consistent weather columns')

In [ ]:
df = pd.read_csv('../data/processed/master_dataset.csv')
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nDtypes:')
print(df.dtypes.to_string())
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())